<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d2-how-a-model-learns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 3 · Notebook 3 — How a model learns

This morning you *were* a neural network: five people, each taking inputs and
following a rule. Today's question is how a network finds its own rules. The
answer has three parts, one per section: a **neuron** is a weighted sum, a
**loss** measures how wrong it is, and **gradient descent** nudges the numbers
until the loss is small. Everything on Day 4 and Day 5 is these three
ideas, scaled up.

Same rules as Day 2: 🔮 predict before you run, ✏️ replace the `...`, ✅ run
the check, 💥 break it when asked. Swap driver and navigator at every `##`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


print("Ready.")

## 1. A neuron is a weighted sum

A neuron takes some inputs, multiplies each by a **weight**, adds them up,
adds a **bias**, and then applies a rule. The most common rule is **ReLU**:
keep the number if it is positive, otherwise output 0.

output = ReLU(w1 × x1 + w2 × x2 + b)

🔮 **Predict.** Inputs `x = [1, 0]`, weights `w = [2, -3]`, bias `b = 0.5`.
What does the neuron output? And for `x = [0, 1]`? Work it out on paper.

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 1a.** Write `neuron(x, w, b)`. `x` and `w` are NumPy arrays of
the same length. (`np.dot(x, w)` multiplies pairs and adds them up;
`max(0, number)` is ReLU.)

In [ ]:
def neuron(x, w, b):
    return ...

In [ ]:
check(is_done(neuron(np.array([1, 0]), np.array([2, -3]), 0.5)), "neuron still returns ... .")
check(np.isclose(neuron(np.array([1, 0]), np.array([2, -3]), 0.5), 2.5), "For x=[1,0]: 2×1 + (−3)×0 + 0.5 = 2.5.")
check(neuron(np.array([0, 1]), np.array([2, -3]), 0.5) == 0, "For x=[0,1] the sum is −2.5, and ReLU turns negatives into 0.")

This morning one neuron in the Human Manual Network followed the rule "up if
either input is up" (OR). Can one neuron with ReLU do that? Here "up" means
the output is above 0.

✏️ **Exercise 1b.** Use the sliders to choose `w1`, `w2` and `b` (run the cell
again after each change) so the neuron outputs 0 for inputs (0, 0) and something above 0 for
the other three.

In [ ]:
w1 = 0.0  # @param {type:"slider", min:-2, max:2, step:0.5}
w2 = 0.0  # @param {type:"slider", min:-2, max:2, step:0.5}
b = 0.0  # @param {type:"slider", min:-2, max:2, step:0.5}
for x in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    out = neuron(np.array(x), np.array([w1, w2]), b)
    print(x, "->", round(out, 2), "up" if out > 0 else "down")

In [ ]:
ups = [neuron(np.array(x), np.array([w1, w2]), b) > 0 for x in [(0, 0), (0, 1), (1, 0), (1, 1)]]
check(ups == [False, True, True, True], f"Up/down pattern is {ups}; OR needs [False, True, True, True].")

🔮 **Predict, then try.** Can you change only `b` to make it AND (up only for
(1, 1))? Try it with the sliders. Then: can any single neuron do XOR (up for
(0, 1) and (1, 0), down for the other two)? Discuss with your partner.

## 2. How wrong is it? The loss

Here is some made-up data: 40 students, the hours each one revised, and their
exam mark.

In [ ]:
rng = np.random.default_rng(7)
hours = rng.uniform(0, 10, 40).round(1)
marks = np.clip(6 * hours + 30 + rng.normal(0, 6, 40), 0, 100).round()

plt.scatter(hours, marks)
plt.xlabel("hours revised")
plt.ylabel("exam mark")
plt.show()

Our model is a straight line: `predicted_mark = w * hours + b`. To say how
good a line is, we need one number for "how wrong": the **mean squared
error** (MSE). For each student, take predicted minus actual, square it, then
average over everyone.

🔮 **Predict.** If every prediction is exactly 2 marks too high, what is the
MSE? And if half are 2 too high and half are 2 too low?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 2a.** Write `mse(predicted, actual)` for two NumPy arrays. One
line, no loop.

In [ ]:
def mse(predicted, actual):
    return ...

In [ ]:
check(is_done(mse(np.array([1.0]), np.array([1.0]))), "mse still returns ... .")
check(np.isclose(mse(np.array([3.0, 5.0]), np.array([1.0, 3.0])), 4), "All 2 too high should give 4.")
check(np.isclose(mse(np.array([3.0, 1.0]), np.array([1.0, 3.0])), 4), "Squaring makes −2 count the same as +2.")
check(np.isclose(mse(np.array([0.0, 0.0]), np.array([0.0, 10.0])), 50), "Average the squared errors: (0 + 100) / 2.")

✏️ **Exercise 2b. Beat the class by hand.** Drag the sliders and re-run the
cell to fit the line.
Get the MSE under 40, then keep going: post your lowest MSE on the class board.

In [ ]:
w_guess = 0.0  # @param {type:"slider", min:0, max:12, step:0.1}
b_guess = 0  # @param {type:"slider", min:0, max:60, step:1}
hand_loss = mse(w_guess * hours + b_guess, marks)
plt.scatter(hours, marks)
plt.plot([0, 10], [b_guess, w_guess * 10 + b_guess], color="C1")
plt.title(f"w = {w_guess}, b = {b_guess}, MSE = {hand_loss:.1f}")
plt.show()

In [ ]:
check(hand_loss < 40, f"MSE is {hand_loss:.1f}. Move the sliders to get under 40.")

## 3. Gradient descent: walk downhill

You fitted the line by looking. A computer can't look, but it can do this:

1. Nudge a number a tiny bit and see whether the loss goes up or down. That
   is the **slope**.
2. Step the other way, downhill. Big slope: big step.
3. Repeat.

Here is the loss written out in full as `loss(w, b)`, so everyone continues
from the same place (compare it with your `mse`). To see gradient descent,
fix `b = 30` and change only `w`. This is the loss for every `w` from 0 to 12.

In [ ]:
def loss(w, b):
    return ((w * hours + b - marks) ** 2).mean()


def loss_of_w(w):
    return loss(w, 30)


ws = np.linspace(0, 12, 200)
plt.plot(ws, [loss_of_w(w) for w in ws])
plt.xlabel("w")
plt.ylabel("loss (MSE)")
plt.show()

🔮 **Predict.** Roughly which `w` gives the lowest loss? If you stand at
`w = 2`, is the slope positive or negative, and which way should you step?

In [ ]:
my_prediction = ""  # @param {type:"string"}

This function measures the slope by nudging `w` by a tiny amount each way.

In [ ]:
def slope(f, w, nudge=1e-4):
    return (f(w + nudge) - f(w - nudge)) / (2 * nudge)


print("slope at w = 2:", round(slope(loss_of_w, 2), 1))
print("slope at w = 10:", round(slope(loss_of_w, 10), 1))

✏️ **Exercise 3a.** Write `step(w, lr)`: one step of gradient descent on
`loss_of_w`. The new `w` is the old `w` minus `lr` times the slope. `lr` is
the **learning rate**: how big a step to take.

In [ ]:
def step(w, lr):
    return ...

In [ ]:
check(is_done(step(2.0, 0.01)), "step still returns ... .")
check(step(2.0, 0.01) > 2.0, "At w = 2 the slope is negative, so a step should make w bigger. Check the sign.")
check(np.isclose(step(2.0, 0.01), 2.0 - 0.01 * slope(loss_of_w, 2.0)), "New w = old w − lr × slope.")

This cell takes 10 steps from `w = 0` and draws the path.

In [ ]:
def descend(lr, steps=10, start=0.0):
    path = [start]
    for _ in range(steps):
        path.append(step(path[-1], lr))
    return path


def draw(path, title):
    plt.plot(ws, [loss_of_w(w) for w in ws], color="lightgrey")
    plt.plot(path, [loss_of_w(w) for w in path], "o-", color="C1")
    plt.ylim(0, 4000)
    plt.title(title)
    plt.show()


path = descend(lr=0.01)
draw(path, "lr = 0.01")
print("w after each step:", np.round(path, 2))

💥 **Break it.** 🔮 First predict: what happens with a learning rate of
`0.035`, a bit more than three times bigger? Then change `0.01` to `0.035` in
the cell below and run it.

In [ ]:
path = descend(lr=0.01)
draw(path, "try a different learning rate")
print("w after each step:", np.round(path, 2))

✏️ **Exercise 3b.** Find a learning rate that gets `w` within 0.05 of the best
value (about 6.2) in **5 steps**.

In [ ]:
my_lr = 0.0  # @param {type:"number"}

In [ ]:
final_w = descend(lr=my_lr, steps=5)[-1]
check(abs(final_w - 6.2) < 0.05, f"After 5 steps w is {final_w:.2f}. Too slow: bigger lr. Bouncing or exploding: smaller.")

## 4. Payoff: let the computer fit the line

Now both numbers at once. At each step we measure the slope for `w` and the
slope for `b` separately, and step both downhill.

🔮 **Predict.** Will gradient descent beat the best MSE on the class board?
How many steps do you think it needs?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
w, b = 0.0, 0.0
lr = 0.02
history = []
for i in range(1000):
    slope_w = slope(lambda t: loss(t, b), w)
    slope_b = slope(lambda t: loss(w, t), b)
    w, b = w - lr * slope_w, b - lr * slope_b
    history.append(loss(w, b))

print(f"after 1000 steps: w = {w:.2f}, b = {b:.2f}, MSE = {history[-1]:.1f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history)
axes[0].set_yscale("log")
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE")
axes[0].set_title("loss going down")
axes[1].scatter(hours, marks)
axes[1].plot([0, 10], [b, w * 10 + b], color="C1")
axes[1].set_title("the fitted line")
plt.show()

✏️ **Exercise 4a.** Find the first step where the loss beat the best hand fit
on the class board. Type the board's best MSE into `board_best`; then make
`first_step_better` the first index `i` where `history[i] < board_best`.
(A loop with `break` works; so does `np.argmax(np.array(history) < board_best)`.)

In [ ]:
board_best = 25.0  # @param {type:"number"}
first_step_better = ...

In [ ]:
check(is_done(first_step_better), "Replace the ... .")
check((np.array(history) < board_best).any(), "Gradient descent never got below that number (its best is about 22.0). Is board_best right?")
check(history[first_step_better] < board_best, "At that step the loss should be below board_best.")
check(first_step_better == 0 or history[first_step_better - 1] >= board_best, "Not the first one: an earlier step was already better.")
print(f"Gradient descent beat the class's best after {first_step_better + 1} steps.")

Gradient descent knows nothing about lines or marks. It only knows how to
measure a slope and step downhill, so it works for any model with numbers in
it. A network on Day 4 has more than 100,000 numbers instead of two, and
PyTorch measures all their slopes at once (that is what `loss.backward()`
does), but each step is the one you just wrote.

## Extension (if you finish early)

Open the Day 3 extension track in Sidecar, or:

- If you know derivatives: the slope of `loss_of_w` is the mean of
  `2 * (w * hours + 30 - marks) * hours`. Check it against `slope`.
- In section 4, plot `w` and `b` against the step number. `w` overshoots to
  about 10 and both crawl back for hundreds of steps. Why so slow? (Hint: plot
  the loss as a contour map over `w` and `b`: it is a long, narrow valley.)
- Fit a curve `w2 * hours**2 + w * hours + b` with the same method.

In [ ]:
# Your extension work here.